In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
cof_data = pd.read_csv('post_dw_coffee_data.csv')

In [3]:
cof_data['Date'] = pd.to_datetime(cof_data['Date'])

In [4]:
coffee_dat = cof_data.drop(columns=['Unnamed: 0'], inplace=False)

In [5]:
# cash_type dropped / Weekdaysort dropped (same info as Weekday) / Monthsort dropped (same info as  Month_name) / Time dropped (same info as hour_of_day
cof_data_mod = coffee_dat.drop(columns=['cash_type', 'Weekdaysort', 'Monthsort', 'Time'])

In [6]:
coff_data_encoded = pd.get_dummies(cof_data_mod, columns=['hour_of_day','coffee_name','Time_of_Day', 'Weekday', 'Month_name'], drop_first=True)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

y = coff_data_encoded['money']
date_column = coff_data_encoded['Date']
data_to_scale = coff_data_encoded.drop(columns=['money', 'Date'])

scaled_array = scaler.fit_transform(data_to_scale)

coff_data_scaled = pd.DataFrame(
    scaled_array, 
    columns=data_to_scale.columns, 
    index=data_to_scale.index
)


In [8]:
coff_data_scaled.columns

Index(['hour_of_day_7', 'hour_of_day_8', 'hour_of_day_9', 'hour_of_day_10',
       'hour_of_day_11', 'hour_of_day_12', 'hour_of_day_13', 'hour_of_day_14',
       'hour_of_day_15', 'hour_of_day_16', 'hour_of_day_17', 'hour_of_day_18',
       'hour_of_day_19', 'hour_of_day_20', 'hour_of_day_21', 'hour_of_day_22',
       'coffee_name_Americano with Milk', 'coffee_name_Cappuccino',
       'coffee_name_Cocoa', 'coffee_name_Cortado', 'coffee_name_Espresso',
       'coffee_name_Hot Chocolate', 'coffee_name_Latte', 'Time_of_Day_Morning',
       'Time_of_Day_Night', 'Weekday_Mon', 'Weekday_Sat', 'Weekday_Sun',
       'Weekday_Thu', 'Weekday_Tue', 'Weekday_Wed', 'Month_name_Aug',
       'Month_name_Dec', 'Month_name_Feb', 'Month_name_Jan', 'Month_name_Jul',
       'Month_name_Jun', 'Month_name_Mar', 'Month_name_May', 'Month_name_Nov',
       'Month_name_Oct', 'Month_name_Sep'],
      dtype='object')

In [9]:
from sklearn.model_selection import train_test_split
X = coff_data_scaled

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 123)

In [10]:
# Testing Linear Regression
from sklearn.model_selection import cross_val_score, KFold 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=123)
linreg = LinearRegression()
linreg_cv = cross_val_score(linreg, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')

linreg.fit(X_train, y_train)

y_pred = linreg.predict(X_test)

print("Linear Reg. Test R2:", r2_score(y_test, y_pred))
print("Linear Reg. Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

print("LR CV RMSE (per fold):", -linreg_cv)
print("LR Mean CV RMSE:", -linreg_cv.mean())

Linear Reg. Test R2: 0.9784956547959586
Linear Reg. Test RMSE: 0.7251792880665924
LR CV RMSE (per fold): [0.68720019 0.75396593 0.83046115 0.71564376 0.70374466]
LR Mean CV RMSE: 0.7382031377421229


In [11]:
#Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=123)

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=123,
    n_jobs=-1
)

rf_cv = cross_val_score(
    rf,
    X_train,
    y_train.values.ravel(),
    cv=kf,
    scoring='neg_root_mean_squared_error'
)

rf.fit(X_train, y_train.values.ravel())
y_pred = rf.predict(X_test)

print("Random Forest Test R2:", r2_score(y_test, y_pred))
print("Random Forest Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("RF CV RMSE (per fold):", -rf_cv)
print("RF Mean CV RMSE:", -rf_cv.mean())


Random Forest Test R2: 0.9744814030859401
Random Forest Test RMSE: 0.7899701406555476
RF CV RMSE (per fold): [0.84734455 0.81567763 0.95562116 0.89398464 0.86254031]
RF Mean CV RMSE: 0.8750336586424273


In [12]:
# Ridge regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

alphas = [0.01, 0.1, 1, 10, 100]

for a in alphas:
    ridge = Ridge(alpha=a)
    
    ridge_cv = cross_val_score(
        ridge,
        X_train,
        y_train.values.ravel(),
        cv=kf,
        scoring='neg_root_mean_squared_error'
    )

    ridge.fit(X_train, y_train.values.ravel())
    y_pred = ridge.predict(X_test)

    print("Alpha: ", a, " Results")
    print("Ridge Test R2:", r2_score(y_test.values.ravel(), y_pred))
    print("Ridge Test RMSE:", np.sqrt(mean_squared_error(y_test.values.ravel(), y_pred)))
    print("Ridge CV RMSE (per fold):", -ridge_cv)
    print("Ridge Mean CV RMSE:", -ridge_cv.mean())
    print()

Alpha:  0.01  Results
Ridge Test R2: 0.9784956754328715
Ridge Test RMSE: 0.7251789401028365
Ridge CV RMSE (per fold): [0.68720462 0.75395871 0.83046209 0.71564189 0.7037485 ]
Ridge Mean CV RMSE: 0.7382031585534553

Alpha:  0.1  Results
Ridge Test R2: 0.9784958421869175
Ridge Test RMSE: 0.7251761284182296
Ridge CV RMSE (per fold): [0.68724505 0.75389415 0.83047094 0.71562571 0.70378352]
Ridge Mean CV RMSE: 0.7382038727329354

Alpha:  1  Results
Ridge Test R2: 0.9784956472946442
Ridge Test RMSE: 0.7251794145479447
Ridge CV RMSE (per fold): [0.68770487 0.75329395 0.8305995  0.71553029 0.70418212]
Ridge Mean CV RMSE: 0.7382621461447919

Alpha:  10  Results
Ridge Test R2: 0.9783221894674361
Ridge Test RMSE: 0.7280982516820935
Ridge CV RMSE (per fold): [0.69723855 0.75123899 0.83552351 0.71996216 0.71245405]
Ridge Mean CV RMSE: 0.743283450765124

Alpha:  100  Results
Ridge Test R2: 0.9672466674860957
Ridge Test RMSE: 0.8949731931879945
Ridge CV RMSE (per fold): [0.96991449 0.91289079 1.04724

In [13]:
from sklearn.linear_model import RidgeCV

alphas = [0.01, 0.1, 1, 10, 100]

ridge_cv_model = RidgeCV(alphas=alphas, cv=kf, scoring='neg_root_mean_squared_error')
ridge_cv_model.fit(X_train, y_train.values.ravel())

y_pred = ridge_cv_model.predict(X_test)

print("Best alpha:", ridge_cv_model.alpha_)
print("RidgeCV Test R2:", r2_score(y_test, y_pred))
print("RidgeCV Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

Best alpha: 0.01
RidgeCV Test R2: 0.9784956754328715
RidgeCV Test RMSE: 0.7251789401028365


**I evaluated Linear Regression, Ridge Regression, and Random Forest models on the coffee shop dataset. Linear Regression and Ridge Regression (with α = 0.01) achieved nearly identical performance, with no meaningful differences in RMSE or R². This indicates that regularization does not improve model performance and suggests that multicollinearity is not a significant issue in the dataset. The Random Forest model performed worse, indicating that the underlying relationships in the data are primarily linear. Therefore, Linear Regression was selected as the final model, as it provides comparable performance while remaining simpler and more interpretable.**

Linear Reg. Test R2: 0.9784956547959586<br>
Linear Reg. Test RMSE: 0.7251792880665924<br>
LR CV RMSE (per fold): [0.68720019 0.75396593 0.83046115 0.71564376 0.70374466]<br>
LR Mean CV RMSE: 0.7382031377421229

Best Alpha:  0.01  Results<br>
Ridge Test R2: 0.9784956754328715<br>
Ridge Test RMSE: 0.7251789401028365<br>
Ridge CV RMSE (per fold): [0.68720462 0.75395871 0.83046209 0.71564189 0.7037485 ]<br>
Ridge Mean CV RMSE: 0.7382031585534553

Random Forest Test R2: 0.9744814030859401<br>
Random Forest Test RMSE: 0.7899701406555475<br>
RF CV RMSE (per fold): [0.84734455 0.81567763 0.95562116 0.89398464 0.86254031]<br>
RF Mean CV RMSE: 0.8750336586424273

**Extract and interpret coefficients from Linear Regression model**

In [14]:
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": linreg.coef_
}).sort_values(by="Coefficient", ascending=False)

print(coefficients)

                            Feature  Coefficient
22                coffee_name_Latte     4.061647
17           coffee_name_Cappuccino     3.392999
21        coffee_name_Hot Chocolate     2.653129
18                coffee_name_Cocoa     2.458426
16  coffee_name_Americano with Milk     2.104802
3                    hour_of_day_10     0.100357
19              coffee_name_Cortado     0.076934
6                    hour_of_day_13     0.049788
5                    hour_of_day_12     0.041446
4                    hour_of_day_11     0.036795
7                    hour_of_day_14     0.030590
9                    hour_of_day_16     0.028749
12                   hour_of_day_19     0.023195
15                   hour_of_day_22     0.020290
0                     hour_of_day_7     0.017706
11                   hour_of_day_18     0.017131
1                     hour_of_day_8     0.015845
24                Time_of_Day_Night     0.015603
26                      Weekday_Sat     0.013083
10                  

**Using Linear Regression Model to Estimate Revenue**

In [15]:
# Total historical revenue
total_revenue = coffee_dat["money"].sum()

# Average monthly revenue
monthly_revenue = (
    coffee_dat.groupby(pd.Grouper(key="Date", freq="M"))["money"]
    .sum()
)

avg_monthly_revenue = monthly_revenue.mean()

print("Total Revenue:", total_revenue)
print("Average Monthly Revenue:", avg_monthly_revenue)

growth_low = 0.02
growth_high = 0.04

target_low = avg_monthly_revenue * (1 + growth_low)
target_high = avg_monthly_revenue * (1 + growth_high)

print("Target Monthly Revenue (2% increase):", target_low)
print("Target Monthly Revenue (4% increase):", target_high)


Total Revenue: 112245.57999999999
Average Monthly Revenue: 8634.275384615385
Target Monthly Revenue (2% increase): 8806.960892307692
Target Monthly Revenue (4% increase): 8979.6464


C:\Users\chill\AppData\Local\Temp\ipykernel_5976\945731241.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  coffee_dat.groupby(pd.Grouper(key="Date", freq="M"))["money"]


In [21]:
# Predict transaction-level revenue
predictions = linreg.predict(X)

# Convert to total predicted revenue
predicted_total_revenue = predictions.sum()

# Convert to monthly estimate
predicted_monthly_revenue = predicted_total_revenue / coffee_dat["Date"].dt.to_period("M").nunique()
print(coffee_dat["Date"].dt.to_period("M").nunique())

print("Predicted Monthly Revenue:", predicted_monthly_revenue)

13
Predicted Monthly Revenue: 8635.81860369378


In [20]:
if predicted_monthly_revenue >= target_low:
    print("✅ 2–4% growth appears achievable based on model predictions.")
else:
    print("⚠️ Growth target may be difficult to achieve under current conditions.")

⚠️ Growth target may be difficult to achieve under current conditions.


In [22]:
print(predicted_total_revenue)

112265.64184801915
